<a href="https://colab.research.google.com/github/S-creator31/Disentangling-Deception-Improving-CNN-Integrity-through-Latent-Feature-Recalibration/blob/main/Runscript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   PyTorch: {torch.__version__}")
else:
    raise RuntimeError("❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU")

✅ GPU: Tesla T4
   VRAM: 15.6 GB
   PyTorch: 2.10.0+cu128


In [2]:
%cd /content
!git clone https://github.com/wkim97/FSR.git
%cd /content/FSR
!echo "--- Repo contents ---" && ls

/content
Cloning into 'FSR'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 55 (delta 7), reused 4 (delta 4), pack-reused 41 (from 1)
Receiving objects: 100% (55/55), 63.71 KiB | 6.37 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/FSR
--- Repo contents ---
advertorch_fsr	environment.yml  LICENSE.txt  models	 run.sh   train.py
attacks		figures		 metric       README.md  test.py


In [3]:
!pip install tqdm --quiet
import tqdm, torchvision
print(f"torchvision : {torchvision.__version__}")
print(f"tqdm        : {tqdm.__version__}")
print("✅ All dependencies OK")

torchvision : 0.25.0+cu128
tqdm        : 4.67.3
✅ All dependencies OK


In [4]:
# ── Bug 1: test.py ── torch.load parenthesis ─────────────────────────────────
with open('test.py', 'r') as f:
    src = f.read()

BUGGY = ("torch.load('./weights/{}/{}/{}.pth'"
         ".format(args.dataset, args.model, args.load_name, map_location=device))")
FIXED = ("torch.load('./weights/{}/{}/{}.pth'"
         ".format(args.dataset, args.model, args.load_name), map_location=device)")

if BUGGY in src:
    src = src.replace(BUGGY, FIXED)
    with open('test.py', 'w') as f:
        f.write(src)
    print("✅ Bug 1 fixed — test.py torch.load parenthesis")
elif FIXED in src:
    print("✅ Bug 1 already correct")
else:
    print("⚠️  Bug 1 pattern not found — inspect test.py manually")


# ── Bug 2: models/BaseModel.py ── abstractclassmethod removed in Python 3.11 ─
with open('models/BaseModel.py', 'r') as f:
    src = f.read()

src = src.replace(
    'from abc import ABC, abstractclassmethod',
    'from abc import ABC, abstractmethod'
)
src = src.replace('@abstractclassmethod', '@abstractmethod')

with open('models/BaseModel.py', 'w') as f:
    f.write(src)
print("✅ Bug 2 fixed — models/BaseModel.py abstractclassmethod → abstractmethod")


# ── Verify both files now import without error ────────────────────────────────
import subprocess, sys

r1 = subprocess.run([sys.executable, '-c',
    'import sys; sys.path.insert(0,"."); from models.BaseModel import BaseModelDNN; print("BaseModel OK")'],
    capture_output=True, text=True, cwd='/content/FSR')
print(r1.stdout.strip() or r1.stderr.strip())

print("\n--- Patched test.py torch.load line ---")
for line in open('test.py'):
    if 'torch.load' in line:
        print(line.rstrip())

✅ Bug 1 fixed — test.py torch.load parenthesis
✅ Bug 2 fixed — models/BaseModel.py abstractclassmethod → abstractmethod
BaseModel OK

--- Patched test.py torch.load line ---
    checkpoint = torch.load('./weights/{}/{}/{}.pth'.format(args.dataset, args.model, args.load_name), map_location=device)


In [5]:
# ─── YOUR SETTINGS ───────────────────────────────────────────────────────────

SAVE_NAME  = 'cifar10_resnet18'  # name for saved checkpoint file
DATASET    = 'cifar10'           # 'cifar10'  |  'svhn'
MODEL      = 'resnet18'          # 'resnet18' | 'vgg16' | 'wideresnet34'
DEVICE     = 0                   # GPU index (keep as 0 in Colab)

NUM_EPOCHS = 5      # ← set to 100 for full paper results (~2.5 hrs)
BATCH_SIZE = 128

# Adversarial training hyperparameters (paper defaults for CIFAR-10)
LR      = 0.1    # use 0.01 for SVHN
EPS     = 8.0    # perturbation budget in units of /255
ALPHA   = 0.25   # PGD step size = ALPHA * EPS/255  |  use 0.125 for SVHN
TAU     = 0.1    # Gumbel-softmax temperature
LAM_SEP = 1.0    # weight on separation loss L_sep
LAM_REC = 1.0    # weight on recalibration loss L_rec

# ─────────────────────────────────────────────────────────────────────────────
print(f"Model      : {MODEL}")
print(f"Dataset    : {DATASET}")
print(f"Epochs     : {NUM_EPOCHS}")
print(f"Batch size : {BATCH_SIZE}")
print(f"LR={LR} | eps={EPS}/255 | alpha={ALPHA} | tau={TAU}")
print(f"lam_sep={LAM_SEP} | lam_rec={LAM_REC}")

Model      : resnet18
Dataset    : cifar10
Epochs     : 5
Batch size : 128
LR=0.1 | eps=8.0/255 | alpha=0.25 | tau=0.1
lam_sep=1.0 | lam_rec=1.0


In [6]:
%cd /content/FSR

!python train.py \
    --save_name {SAVE_NAME} \
    --dataset   {DATASET}   \
    --model     {MODEL}     \
    --device    {DEVICE}    \
    --epoch     {NUM_EPOCHS} \
    --bs        {BATCH_SIZE} \
    --lr        {LR}        \
    --eps       {EPS}       \
    --alpha     {ALPHA}     \
    --tau       {TAU}       \
    --lam_sep   {LAM_SEP}   \
    --lam_rec   {LAM_REC}

/content/FSR
100% 170M/170M [00:03<00:00, 48.7MB/s]

Epoch: 1
cifar10_resnet18 (Train) Epoch: 1/5: : 50000it [06:01, 138.21it/s, Adv_Acc=10.604%, Adv_Loss=2.351, Rec_Loss=2.662, Sep_Loss=4.347]
cifar10_resnet18 (Test) Epoch: 1/5: : 10000it [01:06, 149.72it/s, Adv_Acc=10.670%, Adv_Loss=2.310, Ori_Acc=11.280%, Ori_Loss=2.290]

Epoch: 2
cifar10_resnet18 (Train) Epoch: 2/5: : 50000it [06:02, 137.85it/s, Adv_Acc=10.760%, Adv_Loss=2.300, Rec_Loss=2.548, Sep_Loss=4.116]
cifar10_resnet18 (Test) Epoch: 2/5: : 10000it [01:06, 149.74it/s, Adv_Acc=4.930%, Adv_Loss=2.336, Ori_Acc=13.200%, Ori_Loss=2.298]

Epoch: 3
cifar10_resnet18 (Train) Epoch: 3/5: : 50000it [06:10, 134.96it/s, Adv_Acc=15.932%, Adv_Loss=2.212, Rec_Loss=2.393, Sep_Loss=3.708]
cifar10_resnet18 (Test) Epoch: 3/5: : 10000it [01:06, 150.74it/s, Adv_Acc=18.510%, Adv_Loss=2.165, Ori_Acc=22.020%, Ori_Loss=2.048]

Epoch: 4
cifar10_resnet18 (Train) Epoch: 4/5: : 50000it [06:10, 134.88it/s, Adv_Acc=21.882%, Adv_Loss=2.093, Rec_Loss=2.213, S

In [7]:
%cd /content/FSR

!python test.py \
    --load_name {SAVE_NAME} \
    --dataset   {DATASET}   \
    --model     {MODEL}     \
    --device    {DEVICE}    \
    --tau       {TAU}       \
    --bs        {BATCH_SIZE}

/content/FSR
FGSM: Ori Acc: 36.15%	Adv Acc: 24.57%
PGD-20: Ori Acc: 36.15%	Adv Acc: 24.33%
PGD-100: Ori Acc: 36.15%	Adv Acc: 24.32%
C&W: Ori Acc: 36.15%	Adv Acc: 21.80%
Ensemble : 21.33%


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

src = f'/content/FSR/weights/{DATASET}/{MODEL}/{SAVE_NAME}.pth'
dst_dir = f'/content/drive/MyDrive/FSR_weights'
dst = f'{dst_dir}/{SAVE_NAME}.pth'

os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, dst)
print(f'✅ Weights saved to Google Drive: {dst}')

Mounted at /content/drive
✅ Weights saved to Google Drive: /content/drive/MyDrive/FSR_weights/cifar10_resnet18.pth


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

src = f'/content/drive/MyDrive/FSR_weights/{SAVE_NAME}.pth'
dst_dir = f'/content/FSR/weights/{DATASET}/{MODEL}'
dst = f'{dst_dir}/{SAVE_NAME}.pth'

os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, dst)
print(f'✅ Weights restored to {dst}')
print('You can now run Cell 7 (Evaluate) directly.')

In [ ]:
%cd /content/FSR

# ── VGG16 on CIFAR-10 ──────────────────────────────────────────────────────
# !python train.py --save_name cifar10_vgg16 --dataset cifar10 --model vgg16 --device 0
# !python test.py  --load_name cifar10_vgg16 --dataset cifar10 --model vgg16 --device 0

# ── WideResNet-34-10 on CIFAR-10 ───────────────────────────────────────────
# !python train.py --save_name cifar10_wideresnet34 --dataset cifar10 --model wideresnet34 --device 0
# !python test.py  --load_name cifar10_wideresnet34 --dataset cifar10 --model wideresnet34 --device 0

# ── ResNet-18 on SVHN (different lr and alpha!) ─────────────────────────────
# !python train.py --save_name svhn_resnet18 --dataset svhn --model resnet18 --device 0 --lr 0.01 --alpha 0.125
# !python test.py  --load_name svhn_resnet18 --dataset svhn --model resnet18 --device 0

print("Uncomment a block above and run this cell.")